# VISPR Gemini multimodal — quick start & resume

**First run:** run cells top-to-bottom.

**After a disconnect (resume where it left off):**
1. Re-run **Cell 2** (mount Drive) if the runtime restarted.
2. Re-run **Cell 9** — rebuilds the key pool and re-validates keys (dead/invalid keys are auto-retired; one probe each).
3. Re-run **Cell 14** — it replays the per-sample checkpoints and skips everything already finished.

Nothing already computed is lost: every finished sample is written immediately to `OUTPUT_DIR/checkpoints/*.ckpt.jsonl`. Delete those files only if you want to run a task from scratch.

**When it stops with "All keys are dead or have spent their daily budget":** your 500/day-per-key quota is used up (or the keys are invalid). Wait for the daily reset — or paste fresh keys into **Cell 4** — then resume with steps 2–3 above.

**Key behaviour**
- Each key is capped at `PER_KEY_REQUEST_BUDGET` **actual** requests (retries counted), so you won't exceed the 500/day limit.
- `401/403` → key retired for the run; `429` → key rested for `KEY_COOLDOWN_SEC`, traffic moves to the other keys.
- Speed / scale knobs live in **Cell 4**: `NUM_RUNS`, `TEMPERATURES`, `IMAGE_CACHE_SIZE`, `CONCURRENT_PER_KEY`.

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────
!pip install -q --upgrade google-genai
!pip install -q pillow scikit-learn openpyxl numpy pandas
print("Install complete.")


Install complete.


In [ ]:
# ── Cell 2: Mount Drive & paths ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

JSON_DIR    = "/content/drive/MyDrive/VISPR/test/jsons/test2017"
IMAGE_DIR   = "/content/drive/MyDrive/VISPR/test/images/test2017"
CAPTION_DIR = "/content/drive/MyDrive/001_vispr/new_exps_2/free_select_no_suppression7/captions_v1"
OUTPUT_DIR  = "/content/drive/MyDrive/VISPR/multimodal_results__gemini_flash_3.1_lite_FINAL"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"JSON    : {JSON_DIR}")
print(f"Images  : {IMAGE_DIR}")
print(f"Captions: {CAPTION_DIR}")
print(f"Output  : {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
JSON    : /content/drive/MyDrive/VISPR/test/jsons/test2017
Images  : /content/drive/MyDrive/VISPR/test/images/test2017
Captions: /content/drive/MyDrive/001_vispr/new_exps_2/free_select_no_suppression7/captions_v1
Output  : /content/drive/MyDrive/VISPR/multimodal_results__gemini_flash_3.1_lite_FINAL


In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────
import os, json, re, time, random, threading
from pathlib import Path
from PIL import Image
from collections import Counter
from typing import List, Dict, Set
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from google import genai
from google.genai import types
print("google-genai ready.")


google-genai ready.


In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────
MODEL_ID     = "gemini-3.1-flash-lite"
MODEL_NAME   = "gemini-3.1-flash-lite"
DATASET_NAME = "VISPR-Multimodal"
DATASET_SLUG = "vispr_multimodal"

# ── API keys — paste your Gemini API keys here ─────────────────────
# Invalid/dead keys are detected automatically (one cheap probe each in
# Cell 9) and skipped for the whole run, so a bad key no longer wastes
# # requests or triggers repeated 401/403 errors.
# API_KEYS = [
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
# ]

# API_KEYS = [
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
# ]

# API_KEYS = [
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
# ]

API_KEYS = [
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",
    "AIzaREDACTED_ROTATE_THIS_KEY",

]

# API_KEYS = [
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
# ]

# API_KEYS = [
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AIzaREDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",
#     "AQ.REDACTED_ROTATE_THIS_KEY",

# ]


# API_KEYS = [
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",

# ]

# API_KEYS = [
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
#   "AQ.REDACTED_ROTATE_THIS_KEY",
# ]




CONCURRENT_PER_KEY     = 1      # max simultaneous requests per key
PER_KEY_RPM            = 15
KEY_STRIKE_LIMIT       = 15
PER_KEY_REQUEST_BUDGET = 500
MAX_CONCURRENCY        = len(API_KEYS) * CONCURRENT_PER_KEY   # thread-pool size
PER_KEY_REQUEST_BUDGET = 500    # hard cap on ACTUAL requests each key may send
                                # (retries counted). A key that reaches this is
                                # retired; the run keeps going on the other keys
                                # and only stops when ALL keys are spent/dead.
MAX_RETRIES            = 5      # transient-error (5xx/timeout) retries per request
RETRY_BACKOFF_SEC      = 5      # base backoff (seconds) * attempt #
KEY_COOLDOWN_SEC       = 60     # rest a key this long after a 429 / rate-limit
VALIDATE_KEYS          = True   # probe each key once at startup (1 request/key)
IMAGE_CACHE_SIZE       = 256    # decoded+resized images kept in RAM (speed)

NUM_RUNS         = 3
TEMPERATURES     = [0.1, 1]
MAX_IMAGE_PX     = 1024
MAX_TOKENS_T1    = 10
MAX_TOKENS_T2    = 30
MAX_TOKENS_T3    = 150
MAX_TOKENS_T4    = 30   # binary Yes/No per channel — short output
RUN_SEEDS        = [0, 42, 84]

# ── TASK SELECTION ────────────────────────────────────────────────
# Set True/False to control which tasks run in Cell 14.
# Each task saves its own JSON on completion.
# Re-run Cell 14 after changing these to resume/run specific tasks.
RUN_TASK1 = False
RUN_TASK2 = False
RUN_TASK3 = True
RUN_TASK4 = False

# Subset — None for all ~7995 samples
SUBSET_SIZE = None
SUBSET_SEED = 42

print(f"Model      : {MODEL_ID}")
print(f"Dataset    : {DATASET_NAME}")
print(f"Temps      : {TEMPERATURES}")
print(f"Runs/sample: {NUM_RUNS}  |  Seeds: {RUN_SEEDS}")
print(f"API keys   : {len(API_KEYS)}  x  {CONCURRENT_PER_KEY} concurrent/key  =  {MAX_CONCURRENCY} max in-flight")
print(f"Per-key budget: {PER_KEY_REQUEST_BUDGET} actual requests  (key retired at cap; run stops when all keys spent)")
print(f"Subset     : {SUBSET_SIZE if SUBSET_SIZE else 'all'}")
print(f"Tasks to run: T1={RUN_TASK1} T2={RUN_TASK2} T3={RUN_TASK3} T4={RUN_TASK4}")

Model      : gemini-3.1-flash-lite
Dataset    : VISPR-Multimodal
Temps      : [0.1, 1]
Runs/sample: 3  |  Seeds: [0, 42, 84]
API keys   : 10  x  1 concurrent/key  =  10 max in-flight
Per-key budget: 500 actual requests  (key retired at cap; run stops when all keys spent)
Subset     : all
Tasks to run: T1=False T2=False T3=True T4=False


In [ ]:
# ── Cell 5: Privacy Taxonomy (hierarchical, paper Figure 2) ───────
PRIVACY_TAXONOMY = {
    "Biometric Data":                   {"examples": ["face","fingerprints","audio","iris","gait"]},
    "Children Images":                  {"examples": ["school events","playgrounds"]},
    "Financial Information":            {"examples": ["credit cards","checks","receipts"]},
    "HIPAA Data":                       {"examples": ["medical records","prescriptions","health devices","disabilities"]},
    "Legal Identifiers":                {"examples": ["names","IDs","passports","addresses"]},
    "Digital Identifiers":              {"examples": ["email","phone number","passwords","computer screen content"]},
    "Personal Metadata (Demographics)": {"examples": ["gender","race","age","beliefs","occupation"]},
    "GPS Data":                         {"examples": ["gps data","live location"]},
    "Vehicle Information":              {"examples": ["license plates","vehicle ownership"]},
    "Nudity":                           {"examples": ["nudity","explicit content","adult imagery"]},
    "Violent/Unlawful Actions":         {"examples": ["criminal acts","weapons","vandalism","cigarettes"]},
    "Personal Context":                 {"examples": ["pets","home interior","family gatherings","personal items"]},
    "Location Identifiers":             {"examples": ["location photos","landmarks"]},
    "Background Individuals":           {"examples": ["passerby","bystanders","not clearly visible individuals"]},
}
VALID_CATEGORIES = set(PRIVACY_TAXONOMY.keys())

def get_taxonomy_string():
    lines = ["Taxonomy:"]
    lines.append("1. Biometric Data: -face -fingerprints -audio -iris -gait")
    lines.append("2. Children Images: -school events -playgrounds")
    lines.append("3. PII (Personal Identifiable Information):")
    lines.append("  3a. Financial Information: -credit cards -checks -receipts")
    lines.append("  3b. HIPAA Data: -medical records -prescriptions -health devices -disabilities")
    lines.append("  3c. Legal Identifiers: -names -IDs -passports -addresses")
    lines.append("  3d. Digital Identifiers: -email -phone number -passwords -computer screen content")
    lines.append("  3e. Personal Metadata (Demographics): -gender -race -age -beliefs -occupation")
    lines.append("  3f. GPS Data: -gps data -live location")
    lines.append("  3g. Vehicle Information: -license plates -vehicle ownership")
    lines.append("4. Legal Sensitivity Information:")
    lines.append("  4a. Nudity: -nudity -explicit content -adult imagery")
    lines.append("  4b. Violent/Unlawful Actions: -criminal acts -weapons -vandalism -cigarettes")
    lines.append("5. Personal Life:")
    lines.append("  5a. Personal Context: -pets -home interior -family gatherings -personal items")
    lines.append("  5b. Location Identifiers: -location photos -landmarks")
    lines.append("6. Background Individuals: -passerby -bystanders -not clearly visible individuals")
    return "\n".join(lines)

HIER_TO_CAT = {
    "1":"Biometric Data","2":"Children Images",
    "3a":"Financial Information","3b":"HIPAA Data","3c":"Legal Identifiers",
    "3d":"Digital Identifiers","3e":"Personal Metadata (Demographics)",
    "3f":"GPS Data","3g":"Vehicle Information",
    "4a":"Nudity","4b":"Violent/Unlawful Actions",
    "5a":"Personal Context","5b":"Location Identifiers",
    "6":"Background Individuals",
}
FLAT_INDEX_TO_CAT = {str(i+1): cat for i, cat in enumerate(PRIVACY_TAXONOMY.keys())}
print(f"Taxonomy: {len(PRIVACY_TAXONOMY)} categories")


Taxonomy: 14 categories


In [ ]:
# ── Cell 6: Label → Taxonomy mappings ────────────────────────────
VISPR_TO_TAXONOMY = {
    "a9_face_complete":"Biometric Data","a10_face_partial":"Biometric Data",
    "a7_fingerprint":"Biometric Data","a5_eye_color":"Biometric Data",
    "a6_hair_color":"Biometric Data","a11_tattoo":"Biometric Data",
    "a17_color":"Biometric Data","a2_weight_approx":"Biometric Data",
    "a3_height_approx":"Biometric Data","a4_gender":"Biometric Data",
    "a1_age_approx":"Biometric Data","a16_race":"Biometric Data",
    "a30_credit_card":"Financial Information","a37_receipt":"Financial Information",
    "a39_disability_physical":"HIPAA Data","a41_injury":"HIPAA Data","a43_medicine":"HIPAA Data",
    "a19_name_full":"Legal Identifiers","a20_name_first":"Legal Identifiers",
    "a21_name_last":"Legal Identifiers","a23_birth_city":"Legal Identifiers",
    "a24_birth_date":"Legal Identifiers","a26_handwriting":"Legal Identifiers",
    "a29_ausweis":"Legal Identifiers","a31_passport":"Legal Identifiers",
    "a32_drivers_license":"Legal Identifiers","a33_student_id":"Legal Identifiers",
    "a38_ticket":"Legal Identifiers","a8_signature":"Legal Identifiers",
    "a25_nationality":"Legal Identifiers","a27_marital_status":"Legal Identifiers",
    "a35_mail":"Legal Identifiers","a74_address_current_complete":"Legal Identifiers",
    "a75_address_current_partial":"Legal Identifiers",
    "a78_address_home_complete":"Legal Identifiers","a79_address_home_partial":"Legal Identifiers",
    "a92_email_content":"Digital Identifiers","a97_online_conversation":"Digital Identifiers",
    "a85_username":"Digital Identifiers","a90_email":"Digital Identifiers","a49_phone":"Digital Identifiers",
    "a55_religion":"Personal Metadata (Demographics)","a56_sexual_orientation":"Personal Metadata (Demographics)",
    "a57_culture":"Personal Metadata (Demographics)","a61_opinion_general":"Personal Metadata (Demographics)",
    "a62_opinion_political":"Personal Metadata (Demographics)","a46_occupation":"Personal Metadata (Demographics)",
    "a18_ethnic_clothing":"Personal Metadata (Demographics)","a70_education_history":"Personal Metadata (Demographics)",
    "a82_date_time":"Location Identifiers","a73_landmark":"Location Identifiers",
    "a48_occassion_work":"Personal Context","a60_occassion_personal":"Personal Context",
    "a58_hobbies":"Personal Context","a59_sports":"Personal Context",
    "a64_rel_personal":"Personal Context","a65_rel_social":"Personal Context",
    "a66_rel_professional":"Personal Context","a67_rel_competitors":"Personal Context",
    "a68_rel_spectators":"Background Individuals",
    "a103_license_plate_complete":"Vehicle Information","a104_license_plate_partial":"Vehicle Information",
    "a102_vehicle_ownership":"Vehicle Information",
    "a12_semi_nudity":"Nudity","a13_full_nudity":"Nudity",
    "a99_legal_involvement":"Violent/Unlawful Actions",
}

CAPTION_TO_TAXONOMY = {
    "face_complete":"Biometric Data","face_partial":"Biometric Data",
    "fingerprint":"Biometric Data","eye_color":"Biometric Data",
    "hair_color":"Biometric Data","tattoo":"Biometric Data",
    "color":"Biometric Data","weight_approx":"Biometric Data",
    "height_approx":"Biometric Data","age_approx":"Biometric Data",
    "gender":"Biometric Data","race":"Biometric Data",
    "credit_card":"Financial Information","receipt":"Financial Information",
    "disability_physical":"HIPAA Data","injury":"HIPAA Data","medicine":"HIPAA Data",
    "name_full":"Legal Identifiers","name_first":"Legal Identifiers",
    "name_last":"Legal Identifiers","birth_city":"Legal Identifiers",
    "birth_date":"Legal Identifiers","handwriting":"Legal Identifiers",
    "ausweis":"Legal Identifiers","passport":"Legal Identifiers",
    "drivers_license":"Legal Identifiers","student_id":"Legal Identifiers",
    "mail":"Legal Identifiers","ticket":"Legal Identifiers","signature":"Legal Identifiers",
    "address_current_complete":"Legal Identifiers","address_current_partial":"Legal Identifiers",
    "address_home_complete":"Legal Identifiers","address_home_partial":"Legal Identifiers",
    "nationality":"Legal Identifiers","marital_status":"Legal Identifiers",
    "email_content":"Digital Identifiers","online_conversation":"Digital Identifiers",
    "username":"Digital Identifiers","email":"Digital Identifiers","phone":"Digital Identifiers",
    "religion":"Personal Metadata (Demographics)","sexual_orientation":"Personal Metadata (Demographics)",
    "culture":"Personal Metadata (Demographics)","opinion_general":"Personal Metadata (Demographics)",
    "opinion_political":"Personal Metadata (Demographics)","occupation":"Personal Metadata (Demographics)",
    "ethnic_clothing":"Personal Metadata (Demographics)","education_history":"Personal Metadata (Demographics)",
    "date_time":"Location Identifiers","landmark":"Location Identifiers",
    "occassion_work":"Personal Context","occassion_personal":"Personal Context",
    "hobbies":"Personal Context","sports":"Personal Context",
    "rel_personal":"Personal Context","rel_social":"Personal Context",
    "rel_professional":"Personal Context","rel_competitors":"Personal Context",
    "rel_spectators":"Background Individuals","rel_views":"Background Individuals",
    "license_plate_complete":"Vehicle Information","license_plate_partial":"Vehicle Information",
    "vehicle_ownership":"Vehicle Information",
    "semi_nudity":"Nudity","full_nudity":"Nudity",
    "legal_involvement":"Violent/Unlawful Actions",
}
print(f"VISPR mapping   : {len(VISPR_TO_TAXONOMY)} entries")
print(f"Caption mapping : {len(CAPTION_TO_TAXONOMY)} entries")


VISPR mapping   : 66 entries
Caption mapping : 67 entries


In [ ]:
# ── Cell 7: Prompts ───────────────────────────────────────────────
# Task 1: direct binary detection with caption
# Task 2: taxonomy-guided binary detection with caption
# Task 3: flat attribute recognition (paper-style) with caption
# Task 4: source attribution — image_private vs caption_private (Yes/No each)
#
# All prompts inject caption. Two variants: explicit / no_leak.

PROMPT_TASK1_TEMPLATE = (
    'You are given a social media post consisting of an image and an accompanying caption.\n'
    'Caption: "{caption}"\n\n'
    'Considering both the image and the caption, identify if this post violates any '
    'privacy policies. If yes, say "Yes" otherwise say "No".'
)

def get_prompt_task2(caption: str) -> str:
    tax = get_taxonomy_string()
    return (
        'You are given a social media post consisting of an image and an accompanying caption.\n'
        f'Caption: "{caption}"\n\n'
        'Task: Identify if this post violates any privacy policies from the following taxonomy. '
        'If yes, say that is "Private" otherwise say "Safe".\n'
        f'{tax}\n'
        'Instructions:\n'
        '- Do not justify your answer.\n'
        '- Output ONLY a JSON object. No explanation, no markdown.\n'
        '- Output format: {"privacy_state": ["Safe" or "Private"]}\n\n'
        'Now classify the post and output only the JSON.'
    )

def get_prompt_task3(caption: str) -> str:
    tax = get_taxonomy_string()
    return (
        'You are given a social media post consisting of an image and an accompanying caption.\n'
        f'Caption: "{caption}"\n\n'
        'Task: Identify which privacy categories are violated by this post. '
        'Include all applicable categories found in both the image and the caption.\n'
        f'{tax}\n'
        'Instructions:\n'
        '- Do not justify your answer.\n'
        '- Output ONLY a JSON object. No explanation, no markdown.\n'
        '- Output format: {"categories": [Names of the categories]} or {"categories": ["Safe"]}\n'
        '- Use exact category names from the taxonomy above.\n\n'
        'Now classify the post and output only the JSON.'
    )

def get_prompt_task4(caption: str) -> str:
    tax = get_taxonomy_string()
    return (
        'You are given a social media post consisting of an image and an accompanying caption.\n'
        f'Caption: "{caption}"\n\n'
        'Task: Determine the source of any privacy violation in this post based on the taxonomy.\n'
        f'{tax}\n'
        'Instructions:\n'
        '- Does the IMAGE alone contain private information from the taxonomy? Answer Yes or No.\n'
        '- Does the CAPTION alone contain private information from the taxonomy? Answer Yes or No.\n'
        '- Do not justify your answer.\n'
        '- Output ONLY a JSON object. No explanation, no markdown.\n'
        '- Output format: {"image_private": "Yes" or "No", "caption_private": "Yes" or "No"}\n\n'
        'Now classify and output only the JSON.'
    )

print("Prompts defined.")


Prompts defined.


In [ ]:
# ── Cell 8: Dataset loader ────────────────────────────────────────
# GT design:
# Tasks 1 & 2:
#   explicit → GT always Private (explicit caption has PII by construction)
#   no_leak  → GT from VISPR binary label (clean caption adds nothing)
#
# Task 3:
#   gt_combined_explicit = gt_image | gt_cap_explicit
#   gt_combined_noleak   = gt_image only
#
# Task 4:
#   gt_image_private   = True if gt_image is non-empty
#   gt_caption_private = True if gt_cap_explicit non-empty (explicit) / always False (no_leak)

import os

DATASET_CACHE = os.path.join(OUTPUT_DIR, "dataset_cache.json")

def save_dataset_cache(samples, path):
    serializable = []
    for s in samples:
        s2 = dict(s)
        s2["image_filename"] = os.path.basename(s["image_path"])  # store filename only
        del s2["image_path"]
        for k in ("gt_image","gt_cap_explicit","gt_combined_explicit","gt_combined_noleak"):
            if k in s2 and isinstance(s2[k], set): s2[k] = sorted(list(s2[k]))
        serializable.append(s2)
    with open(path, "w") as f: json.dump(serializable, f)
    print(f"Cache saved → {path}")

def load_dataset_cache(path):
    with open(path) as f: data = json.load(f)
    image_index = {os.path.splitext(e.name)[0]: e.path
                   for e in os.scandir(IMAGE_DIR) if not e.name.startswith(".")}
    for s in data:
        s["image_path"] = image_index[os.path.splitext(s.pop("image_filename"))[0]]
        for k in ("gt_image","gt_cap_explicit","gt_combined_explicit","gt_combined_noleak"):
            if k in s and isinstance(s[k], list): s[k] = set(s[k])
    print(f"Loaded from cache: {len(data)} samples")
    return data

if os.path.exists(DATASET_CACHE):
    dataset = load_dataset_cache(DATASET_CACHE)
else:
    print("No cache found. Running full loader...")

    json_index    = {os.path.splitext(e.name)[0]: e.path
                     for e in os.scandir(JSON_DIR)    if e.name.endswith(".json")}
    image_index   = {os.path.splitext(e.name)[0]: e.path
                     for e in os.scandir(IMAGE_DIR)   if not e.name.startswith(".")}
    caption_index = {os.path.splitext(e.name)[0]: e.path
                     for e in os.scandir(CAPTION_DIR) if e.name.endswith(".json")}
    print(f"  JSON files    : {len(json_index)}")
    print(f"  Image files   : {len(image_index)}")
    print(f"  Caption files : {len(caption_index)}")

    samples = []; missing_img = 0; missing_cap = 0
    total = len([k for k in json_index if k in image_index and k in caption_index])
    for i, (img_id, jf_path) in enumerate(sorted(json_index.items())):
        if img_id not in image_index:   missing_img += 1; continue
        if img_id not in caption_index: missing_cap += 1; continue

        if (i+1) % 500 == 0 or i == 0:
            print(f"  [{len(samples):5d}/{total}] processed...")

        with open(jf_path) as f: vis = json.load(f)
        vis_labels = vis.get("labels", [])
        is_safe    = (vis_labels == ["a0_safe"]) or (vis_labels == [])
        gt_image   = set()
        for lbl in vis_labels:
            if lbl == "a0_safe": continue
            mapped = VISPR_TO_TAXONOMY.get(lbl)
            if mapped: gt_image.add(mapped)

        with open(caption_index[img_id]) as f: cap = json.load(f)
        caps         = cap.get("captions", {})
        cap_explicit = caps.get("explicit", "")
        cap_no_leak  = caps.get("no_leak", "")
        gt_cap_explicit = set()
        for lbl_entry in cap.get("labels", []):
            if lbl_entry.get("source") == "caption_induced":
                mapped = CAPTION_TO_TAXONOMY.get(lbl_entry.get("label",""))
                if mapped: gt_cap_explicit.add(mapped)

        samples.append({
            "id":                      img_id,
            "image_path":              image_index[img_id],
            "is_safe":                 is_safe,
            "caption_explicit":        cap_explicit,
            "caption_no_leak":         cap_no_leak,
            "gt_image":                gt_image,
            "gt_cap_explicit":         gt_cap_explicit,
            "gt_combined_explicit":    gt_image | gt_cap_explicit,
            "gt_combined_noleak":      gt_image,
            "gt_image_private":        len(gt_image) > 0,
            "gt_cap_private_explicit": len(gt_cap_explicit) > 0,
            "gt_cap_private_noleak":   False,
        })

    print(f"Loaded  : {len(samples)} samples")
    print(f"  Missing images   : {missing_img}")
    print(f"  Missing captions : {missing_cap}")
    print(f"  Private (VISPR)  : {sum(not s['is_safe'] for s in samples)}")
    print(f"  Safe (VISPR)     : {sum(s['is_safe'] for s in samples)}")
    print(f"  Caption adds GT  : {sum(len(s['gt_cap_explicit'])>0 for s in samples)} images")

    save_dataset_cache(samples, DATASET_CACHE)
    dataset = samples

print(f"\nDataset ready: {len(dataset)} samples")


In [ ]:
# ── Cell 9: Gemini key pool (RPM-paced, RPD-budgeted, auto-retire) ──
import functools

class AllKeysDown(Exception):
    pass

def classify_error(e):
    status = getattr(e, "code", None) or getattr(e, "status_code", None)
    msg = str(e).lower()
    if status in (401, 403) or "permission_denied" in msg or "api key not valid" in msg \
            or "api_key_invalid" in msg or "unauthenticated" in msg:
        return "auth"
    if status == 429 or "resource_exhausted" in msg or "rate limit" in msg or "quota" in msg:
        return "quota"
    if status == 400 or "invalid_argument" in msg:
        return "client"
    return "transient"

def parse_retry_delay(e, default):
    m = re.search(r'seconds["\':\s]*(\d+)', str(e), re.I)
    return float(m.group(1)) if m else default

class Lease:
    __slots__ = ("key", "client")
    def __init__(self, key, client): self.key = key; self.client = client

class KeyPool:
    def __init__(self, api_keys, per_key_concurrency, per_key_budget, per_key_rpm):
        self.keys     = list(api_keys)
        self.budget   = per_key_budget
        self.interval = 60.0 / max(per_key_rpm, 1)          # min seconds between sends per key
        self.clients  = {k: genai.Client(api_key=k) for k in self.keys}
        self.sema     = {k: threading.BoundedSemaphore(per_key_concurrency) for k in self.keys}
        self.used     = {k: 0     for k in self.keys}       # SUCCESSFUL requests -> counted vs RPD
        self.dead     = {k: False for k in self.keys}
        self.cooldown = {k: 0.0   for k in self.keys}
        self.next_ok  = {k: 0.0   for k in self.keys}       # RPM spacing gate
        self.strikes  = {k: 0     for k in self.keys}       # consecutive 429s
        self._cv      = threading.Condition()

    def _usable(self, k, now):
        return (not self.dead[k]) and self.used[k] < self.budget and self.cooldown[k] <= now
    def _spent(self, k):
        return self.dead[k] or self.used[k] >= self.budget

    def acquire(self, poll=0.1):
        while True:
            with self._cv:
                now = time.time()
                if all(self._spent(k) for k in self.keys):
                    raise AllKeysDown("All keys are invalid or have spent their daily (RPD) budget.")
                ready = [k for k in self.keys if self._usable(k, now) and self.next_ok[k] <= now]
                ready.sort(key=lambda k: self.used[k])       # least-used first
                for k in ready:
                    if self.sema[k].acquire(blocking=False):
                        self.next_ok[k] = now + self.interval
                        return Lease(k, self.clients[k])
                self._cv.wait(timeout=poll)

    def release(self, lease):
        with self._cv:
            self.sema[lease.key].release(); self._cv.notify_all()

    def on_success(self, key):
        with self._cv:
            self.used[key] += 1; self.strikes[key] = 0; self._cv.notify_all()
    def on_consumed(self, key):                              # 400 etc.: used a request, no result
        with self._cv:
            self.used[key] += 1; self._cv.notify_all()
    def on_quota(self, key, cooldown, strike_limit):         # 429: pace, do NOT count vs RPD
        with self._cv:
            self.strikes[key] += 1
            self.cooldown[key] = time.time() + cooldown
            if self.strikes[key] >= strike_limit:
                self.dead[key] = True                        # persistent 429 -> daily-exhausted
            self._cv.notify_all()
    def mark_dead(self, key):
        with self._cv:
            self.dead[key] = True; self._cv.notify_all()
    def note_send(self, key):
        with self._cv:
            self.used[key] += 1

    def usage_report(self):
        return {f"key#{i}": ("DEAD" if self.dead[k] else self.used[k])
                for i, k in enumerate(self.keys)}

key_pool = KeyPool(API_KEYS, CONCURRENT_PER_KEY, PER_KEY_REQUEST_BUDGET, PER_KEY_RPM)
print(f"Configured {len(API_KEYS)} clients | {CONCURRENT_PER_KEY} in-flight/key | "
      f"{PER_KEY_RPM} req/min/key | {PER_KEY_REQUEST_BUDGET} successful/day/key.")

def validate_keys(pool):
    alive = 0
    for i, k in enumerate(pool.keys):
        try:
            pool.clients[k].models.generate_content(
                model=MODEL_ID, contents=["ping"],
                config=types.GenerateContentConfig(max_output_tokens=1))
            pool.note_send(k); print(f"  key#{i:<2} OK"); alive += 1
        except Exception as e:
            kind = classify_error(e)
            if kind == "auth":
                pool.mark_dead(k); print(f"  key#{i:<2} DEAD (invalid/401) — retired")
            elif kind == "quota":
                pool.cooldown[k] = time.time() + KEY_COOLDOWN_SEC
                print(f"  key#{i:<2} rate-limited now (kept)"); alive += 1
            else:
                print(f"  key#{i:<2} error: {type(e).__name__} (kept)"); alive += 1
    print(f"Usable keys: {alive}/{len(pool.keys)}")
    return alive

if VALIDATE_KEYS:
    print("Validating keys ..."); validate_keys(key_pool)
print("Model ready.")

In [ ]:
# ── Cell 10: Inference helpers & parsers ──────────────────────────
import functools

@functools.lru_cache(maxsize=IMAGE_CACHE_SIZE)
def prepare_image(image_path, max_px=MAX_IMAGE_PX):
    """Decode + downscale once, then cache in RAM. The same image is used
    for all NUM_RUNS (and across temps/variants/tasks), so caching avoids
    re-reading it from Drive and re-resizing it every single request —
    the biggest per-request cost after the network call itself."""
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > max_px:
        img.thumbnail((max_px, max_px), Image.Resampling.LANCZOS)
    return img

def call_gemini(image_path, prompt, temperature, max_output_tokens, seed=0):
    img    = prepare_image(image_path)
    config = types.GenerateContentConfig(
        temperature=temperature, max_output_tokens=max_output_tokens, seed=seed)
    transient = 0
    while True:
        lease = key_pool.acquire()
        try:
            resp = lease.client.models.generate_content(
                model=MODEL_ID, contents=[img, prompt], config=config)
            key_pool.on_success(lease.key)                # only a real answer counts vs 500 RPD
            return (resp.text or "").strip()
        except Exception as e:
            kind = classify_error(e)
            if kind == "auth":
                key_pool.mark_dead(lease.key); continue
            if kind == "quota":                            # 429 = short-term rate limit, not RPD
                key_pool.on_quota(lease.key, parse_retry_delay(e, KEY_COOLDOWN_SEC), KEY_STRIKE_LIMIT)
                continue
            if kind == "client":
                key_pool.on_consumed(lease.key); return ""
            transient += 1
            if transient >= MAX_RETRIES:
                print(f"  [WARN] giving up after {MAX_RETRIES} transient errors: {e}")
                return ""
            time.sleep(RETRY_BACKOFF_SEC * transient)
        finally:
            key_pool.release(lease)

def _resolve_cat(c_str, valid_cats=None):
    if valid_cats is None: valid_cats = VALID_CATEGORIES
    c_str = str(c_str).strip()
    if c_str.lower() == "safe": return None
    if c_str in valid_cats: return c_str
    m = re.match(r'^(\d+[a-z]?)\.?\s*(.*)$', c_str)
    if m:
        resolved = HIER_TO_CAT.get(m.group(1).lower())
        if resolved and resolved in valid_cats: return resolved
        name = m.group(2).strip()
        if name in valid_cats: return name
        for cat in valid_cats:
            if cat.lower() == name.lower(): return cat
    if re.match(r'^\d+$', c_str):
        resolved = FLAT_INDEX_TO_CAT.get(c_str)
        if resolved and resolved in valid_cats: return resolved
    for cat in valid_cats:
        if cat.lower() == c_str.lower(): return cat
    return None

def parse_task1(response):
    r = response.lower().strip()
    if re.search(r'\byes\b', r): return "Private"
    if re.search(r'\bno\b',  r): return "Safe"
    if r.startswith('y'):          return "Private"
    return "Safe"

def parse_task2(response):
    try:
        m = re.search(r'\{.*\}', response, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            state = parsed.get("privacy_state", [])
            val   = state[0] if isinstance(state, list) and state else state
            return "Private" if str(val).strip().lower() == "private" else "Safe"
    except Exception: pass
    r = response.lower()
    if "private" in r: return "Private"
    if "safe"    in r: return "Safe"
    return "Safe"

def parse_task3(response):
    predicted = set()
    try:
        m = re.search(r'\{.*\}', response, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            cats = parsed.get("categories", [])
            if isinstance(cats, list):
                for c in cats:
                    if str(c).lower() == "safe": return set()
                    r = _resolve_cat(c)
                    if r: predicted.add(r)
            if predicted: return predicted
    except Exception: pass
    r = response.strip()
    if re.search(r'\bsafe\b', r, re.IGNORECASE): return set()
    for cat in VALID_CATEGORIES:
        if re.search(r'\b' + re.escape(cat) + r'\b', r, re.IGNORECASE):
            predicted.add(cat)
    return predicted

def parse_task4(response):
    # Returns {"image_private": bool, "caption_private": bool}
    try:
        m = re.search(r'\{.*\}', response, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            img_val = str(parsed.get("image_private",   "no")).strip().lower()
            cap_val = str(parsed.get("caption_private", "no")).strip().lower()
            return {
                "image_private":   img_val == "yes",
                "caption_private": cap_val == "yes",
            }
    except Exception: pass
    # Text-scan fallback
    r = response.lower()
    lines = r.split("\n")
    img_private = False; cap_private = False
    for line in lines:
        if "image" in line:
            img_private = "yes" in line
        if "caption" in line:
            cap_private = "yes" in line
    return {"image_private": img_private, "caption_private": cap_private}

def majority_binary(votes): return max(set(votes), key=votes.count)

def majority_labels(all_runs, num_runs):
    counts = Counter(lbl for run in all_runs for lbl in run)
    return {cat for cat, cnt in counts.items() if cnt > num_runs/2}

def majority_task4(all_runs, num_runs):
    img_votes = [r["image_private"]   for r in all_runs]
    cap_votes = [r["caption_private"] for r in all_runs]
    return {
        "image_private":   img_votes.count(True)   > num_runs/2,
        "caption_private": cap_votes.count(True) > num_runs/2,
    }

print("Helpers & parsers defined.")

In [ ]:
# ── Cell 11: Evaluation ───────────────────────────────────────────

def evaluate_detection(results):
    y_true = [1 if r["gt"]=="Private" else 0 for r in results]
    y_pred = [1 if r["prediction"]=="Private" else 0 for r in results]
    acc    = accuracy_score(y_true, y_pred)*100
    p,r,f1,_ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    priv_r = sum(1 for r in results if r["gt"]=="Private" and r["prediction"]=="Private")
    priv_t = sum(1 for r in results if r["gt"]=="Private")
    safe_r = sum(1 for r in results if r["gt"]=="Safe"    and r["prediction"]=="Safe")
    safe_t = sum(1 for r in results if r["gt"]=="Safe")
    return {
        "macro_f1":          round(f1*100,2),
        "macro_precision":   round(p*100,2),
        "macro_recall":      round(r*100,2),
        "accuracy":          round(acc,2),
        "private_recall":    round(100*priv_r/max(priv_t,1),1),
        "safe_recall":       round(100*safe_r/max(safe_t,1),1),
        "private_total":     priv_t,
        "safe_total":        safe_t,
        "predicted_private": sum(1 for r in results if r["prediction"]=="Private"),
        "predicted_safe":    sum(1 for r in results if r["prediction"]=="Safe"),
    }

def evaluate_recognition(results, gt_key, pred_key):
    category_metrics = {}
    for cat in list(VALID_CATEGORIES) + ["Safe"]:
        if cat == "Safe":
            y_true = [1 if len(r[gt_key])==0  else 0 for r in results]
            y_pred = [1 if len(r[pred_key])==0 else 0 for r in results]
        else:
            y_true = [1 if cat in r[gt_key]   else 0 for r in results]
            y_pred = [1 if cat in r[pred_key] else 0 for r in results]
        support = int(sum(y_true))
        if support == 0: continue
        p,r,f1,_ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", pos_label=1, zero_division=0)
        category_metrics[cat] = {
            "precision":round(p*100,2), "recall":round(r*100,2),
            "f1":round(f1*100,2), "support":support}
    f1v = [m["f1"] for m in category_metrics.values()]
    pv  = [m["precision"] for m in category_metrics.values()]
    rv  = [m["recall"]    for m in category_metrics.values()]
    return {
        "category_metrics": category_metrics,
        "macro_f1":         round(np.mean(f1v),2) if f1v else 0.0,
        "macro_precision":  round(np.mean(pv),2)  if pv  else 0.0,
        "macro_recall":     round(np.mean(rv),2)  if rv  else 0.0,
    }

def evaluate_task4_source(results, variant):
    # runner stores gt_cap_private (not variant-suffixed)
    out = {}
    for channel, gt_key, pred_key in [
        ("image",   "gt_image_private",   "pred_image_private"),
        ("caption", "gt_cap_private",     "pred_caption_private"),
    ]:
        y_true = [1 if r[gt_key]   else 0 for r in results]
        y_pred = [1 if r[pred_key] else 0 for r in results]
        support = int(sum(y_true))
        if support == 0:
            out[channel] = {"precision":0,"recall":0,"f1":0,"support":0,
                            "note":"no positive GT"}; continue
        p,r,f1,_ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", pos_label=1, zero_division=0)
        out[channel] = {
            "precision":   round(p*100,2),
            "recall":      round(r*100,2),
            "f1":          round(f1*100,2),
            "support":     support,
            "pred_positive": int(sum(y_pred)),
        }
    return out

print("Evaluation defined.")


In [ ]:
# ── Cell 12: Checkpointing (append-per-sample, crash-safe resume) ──
# Every FINISHED sample is appended to a .jsonl log the instant it is
# done, so a Colab disconnect loses nothing that was already computed
# (the old code only saved every 100 samples -> up to ~300 API calls
# could be thrown away on a drop). On resume the log is replayed and
# completed sample-ids are skipped.
CKPT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

_ckpt_locks       = {}
_ckpt_locks_guard = threading.Lock()
def _ckpt_lock(path):
    with _ckpt_locks_guard:
        if path not in _ckpt_locks:
            _ckpt_locks[path] = threading.Lock()
        return _ckpt_locks[path]

def _ckpt_path(temp, task, variant):
    slug     = MODEL_NAME.replace("/","_").replace(".","_")
    temp_str = str(temp).replace(".","_")
    return os.path.join(CKPT_DIR, f"{slug}_{temp_str}_{task}_{variant}.ckpt.jsonl")

_SET_KEYS = ("gt_image","gt_cap_explicit","gt_combined_explicit","gt_combined_noleak",
             "gt_combined","pred_labels")
def _ser(r):
    rc = dict(r)
    for k in _SET_KEYS:
        if k in rc and isinstance(rc[k], set): rc[k] = sorted(list(rc[k]))
    return rc
def _deser(r):
    rc = dict(r)
    for k in _SET_KEYS:
        if k in rc and isinstance(rc[k], list): rc[k] = set(rc[k])
    return rc

def append_result(r, temp, task, variant):
    """Persist one finished sample immediately (thread-safe, durable)."""
    path = _ckpt_path(temp, task, variant)
    line = json.dumps(_ser(r))
    with _ckpt_lock(path):
        with open(path, "a") as f:
            f.write(line + "\n")
            f.flush()
            try: os.fsync(f.fileno())      # force to disk; Drive-FUSE may No-op
            except OSError: pass

def load_checkpoint(temp, task, variant):
    path = _ckpt_path(temp, task, variant)
    if not os.path.exists(path): return []
    by_id = {}
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln: continue
            try:
                r = json.loads(ln)
            except Exception:
                continue                    # ignore a half-written final line
            by_id[r["id"]] = _deser(r)      # last write wins (dedupes reruns)
    results = list(by_id.values())
    if results: print(f"  Resumed {task}/{variant}: {len(results)} done")
    return results

print(f"Checkpoint dir: {CKPT_DIR}")

In [ ]:
# ── Cell 13: Task runners (concurrent, stops cleanly when pool spent) ─
# Each task fires NUM_RUNS x len(remaining) requests through a
# ThreadPoolExecutor. call_gemini()/KeyPool pick a healthy key per
# request (round-robin over keys that still have budget and aren't dead
# or cooling), so work spreads across every live key automatically.
#
# Every finished sample is checkpointed the instant it completes
# (append_result), so a disconnect loses nothing already done. If a
# sample's request hits AllKeysDown (no key left to serve), that sample
# is simply left unfinished -> absent from the checkpoint -> retried in
# full next time. run_task* returns (results, elapsed, stopped);
# "stopped" tells Cell 14 to halt the pipeline.

def _run_task(samples, temp, variant, task_name, max_tokens, prompt_fn,
              parse_fn, finalize_fn):
    """Shared driver for all four tasks. prompt_fn(sample)->prompt,
    parse_fn(raw)->parsed, finalize_fn(sample, parsed_runs, raw_runs)->record."""
    results   = load_checkpoint(temp, task_name, variant)
    done_ids  = {r["id"] for r in results}
    remaining = [s for s in samples if s["id"] not in done_ids]
    by_id     = {s["id"]: s for s in remaining}
    t_start   = time.time()

    jobs = []  # (sample_id, run_idx, prompt)
    for s in remaining:
        prompt = prompt_fn(s)
        for run in range(NUM_RUNS):
            jobs.append((s["id"], run, prompt))

    parsed_by_sample = {sid: [None]*NUM_RUNS for sid in by_id}
    raw_by_sample    = {sid: [None]*NUM_RUNS for sid in by_id}
    pending          = Counter({sid: NUM_RUNS for sid in by_id})
    n_done_samples   = 0
    stopped          = False

    def worker(job):
        sid, run, prompt = job
        s   = by_id[sid]
        raw = call_gemini(s["image_path"], prompt, temp, max_tokens, RUN_SEEDS[run])
        return sid, run, raw

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENCY) as ex:
        futures = [ex.submit(worker, job) for job in jobs]
        for fut in as_completed(futures):
            try:
                sid, run, raw = fut.result()
            except AllKeysDown as e:
                if not stopped:
                    stopped = True
                    print(f"\n  [STOP] {e}")
                    print(f"  Halting — {len(results)} samples completed and checkpointed so far.")
                continue
            raw_by_sample[sid][run]    = raw
            parsed_by_sample[sid][run] = parse_fn(raw)
            pending[sid] -= 1
            if pending[sid] == 0:
                rec = finalize_fn(by_id[sid], parsed_by_sample[sid], raw_by_sample[sid])
                results.append(rec)
                append_result(rec, temp, task_name, variant)   # persist immediately
                n_done_samples += 1
                if n_done_samples % 10 == 0 or n_done_samples == 1:
                    el = time.time() - t_start
                    print(f"  [{len(results):5d}/{len(samples)}]  {el:.0f}s  avg {el/n_done_samples:.1f}s/img")

    if stopped: print(f"  Key usage at stop: {key_pool.usage_report()}")
    return results, time.time() - t_start, stopped

def _cap(s, variant):
    return s["caption_explicit"] if variant == "explicit" else s["caption_no_leak"]
def _binary_gt(s, variant):
    return "Private" if variant == "explicit" else ("Safe" if s["is_safe"] else "Private")

def run_task1(samples, temp, variant):
    def finalize(s, parsed_runs, raw_runs):
        return {"id": s["id"], "gt": _binary_gt(s, variant),
                "prediction": majority_binary(parsed_runs),
                "all_runs": parsed_runs, "raw_outputs": raw_runs}
    return _run_task(samples, temp, variant, "task1", MAX_TOKENS_T1,
                     lambda s: PROMPT_TASK1_TEMPLATE.format(caption=_cap(s, variant)),
                     parse_task1, finalize)

def run_task2(samples, temp, variant):
    def finalize(s, parsed_runs, raw_runs):
        return {"id": s["id"], "gt": _binary_gt(s, variant),
                "prediction": majority_binary(parsed_runs),
                "all_runs": parsed_runs, "raw_outputs": raw_runs}
    return _run_task(samples, temp, variant, "task2", MAX_TOKENS_T2,
                     lambda s: get_prompt_task2(_cap(s, variant)),
                     parse_task2, finalize)

def run_task3(samples, temp, variant):
    def finalize(s, parsed_runs, raw_runs):
        gt_combined = s["gt_combined_explicit"] if variant == "explicit" else s["gt_combined_noleak"]
        final = majority_labels(parsed_runs, NUM_RUNS)
        return {"id": s["id"], "gt_combined": gt_combined, "pred_labels": final,
                "all_runs": [sorted(list(r)) for r in parsed_runs],
                "raw_outputs": raw_runs}
    return _run_task(samples, temp, variant, "task3", MAX_TOKENS_T3,
                     lambda s: get_prompt_task3(_cap(s, variant)),
                     parse_task3, finalize)

def run_task4(samples, temp, variant):
    def finalize(s, parsed_runs, raw_runs):
        gt_img_priv = s["gt_image_private"]
        gt_cap_priv = s["gt_cap_private_explicit"] if variant == "explicit" else s["gt_cap_private_noleak"]
        final = majority_task4(parsed_runs, NUM_RUNS)
        return {
            "id":                   s["id"],
            "gt_image_private":     gt_img_priv,
            "gt_cap_private":       gt_cap_priv,
            "pred_image_private":   final["image_private"],
            "pred_caption_private": final["caption_private"],
            "all_runs":             parsed_runs,
            "raw_outputs":          raw_runs,
        }
    return _run_task(samples, temp, variant, "task4", MAX_TOKENS_T4,
                     lambda s: get_prompt_task4(_cap(s, variant)),
                     parse_task4, finalize)

print("Task runners defined.")

In [ ]:
# ── Cell 14: MAIN PIPELINE ────────────────────────────────────────
# TASK CONTROL: set RUN_TASK1/2/3/4 in Cell 4.
# Each completed task saves its own JSON immediately after finishing.
# STOP-ON-BUDGET: if any key hits PER_KEY_REQUEST_BUDGET requests, the
# active task stops cleanly, checkpoints whatever finished, and this
# whole loop halts (remaining variants/temps/tasks are skipped).
# RESUME: paste fresh keys into API_KEYS (Cell 4), re-run Cell 9,
# then re-run this cell — checkpoints skip already-completed samples.

slug = MODEL_NAME.replace("/","_").replace(".","_")

def save_task_json(task_key, task_data, temp_results=None):
    ts   = time.strftime("%Y%m%d_%H%M%S")
    path = os.path.join(OUTPUT_DIR, f"{slug}_{DATASET_SLUG}_{task_key}_{ts}.json")
    ser  = {}
    for tk, tvars in task_data.items():
        ser[tk] = {}
        for variant, vdata in tvars.items():
            rc = []
            for r in vdata["results"]:
                r2 = dict(r)
                for k in ("gt_image","gt_cap_explicit","gt_combined_explicit",
                          "gt_combined_noleak","gt_combined","pred_labels",
                          "pred_combined","pred_image","pred_caption"):
                    if k in r2 and isinstance(r2[k], set): r2[k] = sorted(list(r2[k]))
                rc.append(r2)
            ser[tk][variant] = {"metrics":vdata["metrics"],"elapsed":vdata["elapsed"],
                                "n_samples":len(rc),"results":rc}
    ser["_meta"] = {"model_id":MODEL_ID,"dataset":DATASET_NAME,"task":task_key,
                    "n_samples":len(dataset),"temperatures":TEMPERATURES,
                    "num_runs":NUM_RUNS,"seeds":RUN_SEEDS,"timestamp":ts}
    with open(path,"w") as f: json.dump(ser,f,indent=2)
    print(f"  JSON saved → {path}")
    return path

print("\n"+"="*70)
print(f" MODEL   : {MODEL_ID}")
print(f" DATASET : {DATASET_NAME}  ({len(dataset)} samples)")
print(f" TEMPS   : {TEMPERATURES}")
print(f" Running : T1={RUN_TASK1}  T2={RUN_TASK2}  T3={RUN_TASK3}  T4={RUN_TASK4}")
print("="*70+"\n")

all_results = {"task1":{},"task2":{},"task3":{},"task4":{}}
pipeline_start = time.time()
pipeline_stopped = False

for temp in TEMPERATURES:
    if pipeline_stopped: break
    tk_ = f"temp={temp}"
    print(f"\n{'─'*70}\nTEMPERATURE: {temp}\n{'─'*70}")

    for variant in ["explicit","no_leak"]:
        if pipeline_stopped: break

        if RUN_TASK1:
            print(f"\n▶ Task 1 [{variant}]  (temp={temp})")
            r1,e1,stopped1 = run_task1(dataset,temp,variant)
            m1    = evaluate_detection(r1)
            all_results["task1"].setdefault(tk_,{})[variant]={"results":r1,"metrics":m1,"elapsed":e1}
            print(f"   Macro F1: {m1['macro_f1']}%  P: {m1['macro_precision']}%  R: {m1['macro_recall']}%")
            print(f"   Private recall: {m1['private_recall']}%  Safe recall: {m1['safe_recall']}%")
            if stopped1: pipeline_stopped = True

        if RUN_TASK2 and not pipeline_stopped:
            print(f"\n▶ Task 2 [{variant}]  (temp={temp})")
            r2,e2,stopped2 = run_task2(dataset,temp,variant)
            m2    = evaluate_detection(r2)
            all_results["task2"].setdefault(tk_,{})[variant]={"results":r2,"metrics":m2,"elapsed":e2}
            print(f"   Macro F1: {m2['macro_f1']}%  P: {m2['macro_precision']}%  R: {m2['macro_recall']}%")
            print(f"   Private recall: {m2['private_recall']}%  Safe recall: {m2['safe_recall']}%")
            if stopped2: pipeline_stopped = True

        if RUN_TASK3 and not pipeline_stopped:
            print(f"\n▶ Task 3 [{variant}]  (temp={temp})")
            r3,e3,stopped3 = run_task3(dataset,temp,variant)
            m3    = evaluate_recognition(r3,"gt_combined","pred_labels")
            all_results["task3"].setdefault(tk_,{})[variant]={"results":r3,"metrics":m3,"elapsed":e3}
            print(f"   Macro F1: {m3['macro_f1']}%  P: {m3['macro_precision']}%  R: {m3['macro_recall']}%")
            for cat,cm in sorted(m3["category_metrics"].items(),key=lambda x:x[1]["f1"],reverse=True):
                print(f"     {cat:<40} F1={cm['f1']:5.1f}%  P={cm['precision']:5.1f}%  R={cm['recall']:5.1f}%  n={cm['support']}")
            if stopped3: pipeline_stopped = True

        if RUN_TASK4 and not pipeline_stopped:
            print(f"\n▶ Task 4 [{variant}]  (temp={temp})")
            r4,e4,stopped4 = run_task4(dataset,temp,variant)
            m4    = evaluate_task4_source(r4,variant)
            all_results["task4"].setdefault(tk_,{})[variant]={"results":r4,"metrics":m4,"elapsed":e4}
            for ch,cm in m4.items():
                print(f"   {ch.upper()} channel — F1: {cm['f1']}%  P: {cm['precision']}%  R: {cm['recall']}%  support={cm['support']}")
            if stopped4: pipeline_stopped = True

# Save per-task JSONs immediately (whatever finished — partial or full)
print("\n── Saving task JSONs ─────────────────────────────────────────")
if RUN_TASK1 and all_results["task1"]: save_task_json("task1", all_results["task1"], all_results)
if RUN_TASK2 and all_results["task2"]: save_task_json("task2", all_results["task2"], all_results)
if RUN_TASK3 and all_results["task3"]: save_task_json("task3", all_results["task3"], all_results)
if RUN_TASK4 and all_results["task4"]: save_task_json("task4", all_results["task4"], all_results)

pe = time.time()-pipeline_start
print(f"\nTotal: {pe:.0f}s  ({pe/60:.1f} min)")
if pipeline_stopped:
    print("STOPPED EARLY — a key hit its request budget.")
    print("Paste fresh keys into API_KEYS (Cell 4), re-run Cell 9, then re-run this cell to resume.")
print("="*70)


In [ ]:
import time
now = time.time()
for i, k in enumerate(key_pool.keys):
    print(f"key#{i:<2} used={key_pool.used[k]:<4} dead={key_pool.dead[k]!s:<5} "
          f"strikes={key_pool.strikes[k]:<3} cooldown={max(0, key_pool.cooldown[k]-now):.0f}s")

In [ ]:
# ── Cell 15: Save combined JSON + Excel ───────────────────────────
slug      = MODEL_NAME.replace("/","_").replace(".","_")
timestamp = time.strftime("%Y%m%d_%H%M%S")

def serialize_result(r):
    r2 = dict(r)
    for k, v in r2.items():
        if isinstance(v, set): r2[k] = sorted(list(v))
    return r2

# Combined JSON
json_path = os.path.join(OUTPUT_DIR, f"{slug}_{DATASET_SLUG}_{timestamp}_all_tasks.json")
combined  = {}
for task_key, task_data in all_results.items():
    if not task_data: continue
    combined[task_key] = {}
    for tk_, tvars in task_data.items():
        combined[task_key][tk_] = {}
        for variant, vdata in tvars.items():
            rc = [serialize_result(r) for r in vdata["results"]]
            combined[task_key][tk_][variant] = {
                "metrics":   vdata["metrics"],
                "elapsed":   vdata["elapsed"],
                "n_samples": len(rc),
                "results":   rc,
            }
combined["_meta"] = {
    "model_id":    MODEL_ID, "dataset": DATASET_NAME, "slug": DATASET_SLUG,
    "n_samples":   len(dataset), "subset_size": SUBSET_SIZE, "subset_seed": SUBSET_SEED,
    "temperatures":TEMPERATURES, "num_runs": NUM_RUNS, "seeds": RUN_SEEDS,
    "timestamp":   timestamp,
    "taxonomy_prompt": "hierarchical (paper Figure 2 format)",
    "gt_design": {
        "task1_task2_explicit": "Always Private",
        "task1_task2_no_leak":  "VISPR binary label",
        "task3_gt_combined_explicit": "gt_image | gt_cap_explicit",
        "task3_gt_combined_noleak":   "gt_image only",
        "task4_gt_image_private":     "True if gt_image non-empty",
        "task4_gt_caption_private":   "True if gt_cap_explicit non-empty (explicit) / Always False (no_leak)",
    },
}
with open(json_path,"w") as f: json.dump(combined,f,indent=2)
print(f"Combined JSON → {json_path}")

# Excel
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
wb=Workbook(); wb.remove(wb.active)
HDR=PatternFill("solid",fgColor="1F3864"); BEST=PatternFill("solid",fgColor="E2EFDA")
SUB=PatternFill("solid",fgColor="D6E4F0")
HF=Font(color="FFFFFF",bold=True,size=11)
THIN=Side(style="thin"); BDR=Border(left=THIN,right=THIN,top=THIN,bottom=THIN)
C=Alignment(horizontal="center",vertical="center",wrap_text=True)
L=Alignment(horizontal="left",vertical="center",wrap_text=True)
def hdr(ws,row,col,val,w=None):
    ce=ws.cell(row=row,column=col,value=val); ce.fill=HDR; ce.font=HF; ce.border=BDR; ce.alignment=C
    if w: ws.column_dimensions[get_column_letter(col)].width=w
def cel(ws,row,col,val,bold=False,fill=None,align=C):
    ce=ws.cell(row=row,column=col,value=val); ce.font=Font(bold=bold)
    ce.border=BDR; ce.alignment=align
    if fill: ce.fill=fill

# Sheet 1 — Detection T1+T2
ws1=wb.create_sheet("Detection T1+T2")
for col,(label,w) in enumerate([("Task",10),("Variant",12),("Temp",7),
    ("Macro F1",11),("Macro P",11),("Macro R",11),("Accuracy",11),
    ("Private Recall",14),("Safe Recall",12),("Pred Private",13),("Pred Safe",12)],1):
    hdr(ws1,1,col,label,w)
row=2
for task_key,label in [("task1","Task 1"),("task2","Task 2")]:
    for tk_,tvars in all_results.get(task_key,{}).items():
        tv=tk_.replace("temp=","")
        for variant,vdata in tvars.items():
            m=vdata["metrics"]
            cel(ws1,row,1,label,align=L); cel(ws1,row,2,variant); cel(ws1,row,3,tv)
            cel(ws1,row,4,m["macro_f1"],bold=True)
            cel(ws1,row,5,m.get("macro_precision","-")); cel(ws1,row,6,m.get("macro_recall","-"))
            cel(ws1,row,7,m.get("accuracy","-"))
            cel(ws1,row,8,m.get("private_recall","-")); cel(ws1,row,9,m.get("safe_recall","-"))
            cel(ws1,row,10,m.get("predicted_private","-")); cel(ws1,row,11,m.get("predicted_safe","-"))
            row+=1

# Sheet 2 — Recognition T3
ws2=wb.create_sheet("Recognition T3")
for col,(label,w) in enumerate([("Variant",12),("Temp",7),("Category",32),
    ("Precision %",12),("Recall %",12),("F1 %",11),("Support",9)],1):
    hdr(ws2,1,col,label,w)
row=2
for tk_,tvars in all_results.get("task3",{}).items():
    tv=tk_.replace("temp=","")
    for variant,vdata in tvars.items():
        m=vdata["metrics"]
        for cat,cm in sorted(m["category_metrics"].items(),key=lambda x:x[1]["f1"],reverse=True):
            fill=SUB if cat=="Safe" else None
            cel(ws2,row,1,variant,fill=fill); cel(ws2,row,2,tv,fill=fill)
            cel(ws2,row,3,cat,fill=fill,align=L)
            cel(ws2,row,4,cm["precision"],fill=fill); cel(ws2,row,5,cm["recall"],fill=fill)
            cel(ws2,row,6,cm["f1"],bold=True,fill=fill); cel(ws2,row,7,cm["support"],fill=fill)
            row+=1
        cel(ws2,row,3,"MACRO",bold=True,fill=BEST,align=L)
        cel(ws2,row,4,m["macro_precision"],bold=True,fill=BEST)
        cel(ws2,row,5,m["macro_recall"],bold=True,fill=BEST)
        cel(ws2,row,6,m["macro_f1"],bold=True,fill=BEST); row+=2

# Sheet 3 — Source Attribution T4
ws3=wb.create_sheet("Source Attribution T4")
for col,(label,w) in enumerate([("Variant",12),("Temp",7),("Channel",12),
    ("Precision %",12),("Recall %",12),("F1 %",11),("Support",9),("Pred Positive",13)],1):
    hdr(ws3,1,col,label,w)
row=2
for tk_,tvars in all_results.get("task4",{}).items():
    tv=tk_.replace("temp=","")
    for variant,vdata in tvars.items():
        m=vdata["metrics"]
        for channel in ["image","caption"]:
            cm=m.get(channel,{})
            cel(ws3,row,1,variant); cel(ws3,row,2,tv)
            cel(ws3,row,3,channel,align=L)
            cel(ws3,row,4,cm.get("precision","-")); cel(ws3,row,5,cm.get("recall","-"))
            cel(ws3,row,6,cm.get("f1","-"),bold=True); cel(ws3,row,7,cm.get("support","-"))
            cel(ws3,row,8,cm.get("pred_positive","-")); row+=1

excel_path=os.path.join(OUTPUT_DIR,f"{slug}_{DATASET_SLUG}_{timestamp}_all_tasks.xlsx")
wb.save(excel_path)
print(f"Excel saved → {excel_path}")
print("\nAll done.")
